# Convolutional Neural Networks (CNN)

A refresher on the architecture that made deep learning work for images: **local receptive fields**, **weight sharing**, and **spatial pooling**.

**Domain:** Architectures  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

A CNN is a neural network built around the **convolution** operation: a small learnable filter (kernel) is slid across the input, computing a dot product at every position to produce a *feature map*. Stack these layers and the network learns a hierarchy of features — edges → textures → object parts → objects.

**The problem it solves.** A 224×224 RGB image is ~150K numbers. A fully-connected layer mapping that to even 1000 hidden units needs ~150M weights — too many to train, and it throws away the one thing we know about images: *nearby pixels are related, and a cat is a cat wherever it appears*. CNNs bake two priors into the architecture instead of forcing the model to learn them from data:

- **Locality** — each neuron looks only at a small patch (its *receptive field*), not the whole image.
- **Translation equivariance via weight sharing** — the *same* filter is reused at every location, so an edge detector learned in one corner works everywhere. This slashes parameter counts by orders of magnitude and gives strong sample efficiency.

**Reach for a CNN when** your data has a grid/spatial structure with local correlations: images (2D), audio spectrograms and time series (1D), volumetric/medical scans (3D). It is still the default, most efficient choice for image classification, detection, and segmentation, and it is the backbone inside many larger systems.

**Don't reach for it when** the input has no spatial locality (tabular features, set data) or when you need to model very long-range global dependencies and have lots of data — there a Transformer/Vision Transformer often wins. See [`resnet`](resnet.ipynb), [`u-net`](u-net.ipynb), and [`vision-transformer`](vision-transformer.ipynb).

## 2. Mental Model

Think of a **convolution as a sliding stamp**. You have a small stencil (the kernel, e.g. 3×3). You press it onto the top-left of the image, multiply-and-sum the overlapping pixels into a single number, slide it one step right, stamp again — covering the whole image. The grid of numbers you stamp out is a **feature map**: bright where the image locally matches the stencil's pattern.

```
input (5x5)        kernel (3x3)        feature map (3x3)
. . . . .                              s s s
. ┌─────┐ .          ┌─────┐           s s s     each s = sum(kernel * patch)
. │ 3x3 │ .    *     │ w w │    =       s s s
. └─────┘ .          │ w w │
. . . . .            └─────┘
   slide the window over every position; reuse the SAME w's everywhere
```

Two more ideas complete the picture:

- **Depth = a stack of stamps.** A conv layer has many filters; each produces one feature-map channel. Early layers stamp out edges and color blobs; deeper layers stamp out combinations of those.
- **Pooling = zoom out.** After convolving, downsample (e.g. take the max of each 2×2 block). This shrinks the maps, builds tolerance to small shifts, and lets the next layer's small kernel cover a *larger* area of the original image (a growing **receptive field**).

## 3. Key Concepts

- **Kernel / filter** — small weight matrix (3×3, 5×5) slid over the input. A conv layer has `out_channels` filters, each spanning all `in_channels`. Learnable params per layer: `K·K·C_in·C_out + C_out` (bias).
- **Feature map (activation map)** — the output of applying one filter across the input; one per output channel.
- **Stride** — step size of the slide. Stride 2 halves the spatial resolution (downsampling without pooling).
- **Padding** — ring of (usually zero) pixels added around the border so the kernel can sit on edge pixels. `padding='same'` keeps the spatial size; `'valid'` (no padding) shrinks it.
- **Output size** — for input `W`, kernel `K`, padding `P`, stride `S`:  `out = floor((W + 2P − K) / S) + 1`. Worth memorizing — it's the source of most shape bugs.
- **Receptive field** — the region of the *input* that influences one output unit. Grows with depth, larger kernels, stride, and pooling. Deep layers "see" most of the image.
- **Pooling** — fixed (non-learned) downsampling. **Max pool** keeps the strongest activation in each window; **average pool** takes the mean. **Global average pooling** collapses each channel to one number, replacing giant final dense layers.
- **Weight sharing & translation equivariance** — the same kernel everywhere ⇒ shift the input, the feature map shifts the same way. The core inductive bias.
- **Channels** — depth dimension. Images start at 3 (RGB); hidden layers have many (64, 128, 256…), each a learned feature type.
- **1×1 convolution** — mixes channels at each pixel without looking at neighbors; cheap way to change channel count (used in bottlenecks, Inception, ResNet).

## 4. Setup

The worked examples below implement convolution and pooling **from scratch in NumPy** — no GPU, no downloads, runs in a second. The final example shows the idiomatic **PyTorch** `nn.Conv2d` API; it is gated behind an environment variable so the notebook executes top-to-bottom whether or not Torch is installed.

```bash
pip install numpy            # required for the runnable cells
pip install torch            # optional: only for the gated PyTorch example
```

In [1]:
import numpy as np

np.set_printoptions(precision=2, suppress=True, linewidth=120)
print("numpy", np.__version__, "— CNN internals from scratch, CPU-only, no downloads")

numpy 2.5.0 — CNN internals from scratch, CPU-only, no downloads


## 5. Worked Examples

### Example 1 — Convolution as a feature detector

We build a tiny synthetic image with a **vertical edge** (dark left half, bright right half) and convolve it with a hand-crafted vertical-edge (Sobel) kernel. The feature map lights up exactly where the edge is — this is what a *learned* first-layer filter ends up doing on its own.

In [2]:
def conv2d(image, kernel, stride=1, padding=0):
    """Single-channel 2D cross-correlation (what DL libraries call 'convolution')."""
    if padding:
        image = np.pad(image, padding, mode="constant")
    H, W = image.shape
    kH, kW = kernel.shape
    outH = (H - kH) // stride + 1
    outW = (W - kW) // stride + 1
    out = np.zeros((outH, outW))
    for i in range(outH):
        for j in range(outW):
            patch = image[i*stride:i*stride+kH, j*stride:j*stride+kW]
            out[i, j] = np.sum(patch * kernel)        # <- same kernel reused at every (i, j)
    return out

# 8x8 image: dark (0) left half, bright (1) right half  ->  one vertical edge in the middle
image = np.zeros((8, 8))
image[:, 4:] = 1.0

sobel_x = np.array([[-1, 0, 1],                        # vertical-edge detector
                    [-2, 0, 2],
                    [-1, 0, 1]])

feature_map = conv2d(image, sobel_x, padding=1)
print("input image:\n", image)
print("\nedge feature map (large |value| = strong vertical edge):\n", feature_map)
print("\nstrongest response at column index:", np.argmax(np.abs(feature_map).sum(axis=0)),
      " (the edge sits between columns 3 and 4)")

input image:
 [[0. 0. 0. 0. 1. 1. 1. 1.]
 [0. 0. 0. 0. 1. 1. 1. 1.]
 [0. 0. 0. 0. 1. 1. 1. 1.]
 [0. 0. 0. 0. 1. 1. 1. 1.]
 [0. 0. 0. 0. 1. 1. 1. 1.]
 [0. 0. 0. 0. 1. 1. 1. 1.]
 [0. 0. 0. 0. 1. 1. 1. 1.]
 [0. 0. 0. 0. 1. 1. 1. 1.]]

edge feature map (large |value| = strong vertical edge):
 [[ 0.  0.  0.  3.  3.  0.  0. -3.]
 [ 0.  0.  0.  4.  4.  0.  0. -4.]
 [ 0.  0.  0.  4.  4.  0.  0. -4.]
 [ 0.  0.  0.  4.  4.  0.  0. -4.]
 [ 0.  0.  0.  4.  4.  0.  0. -4.]
 [ 0.  0.  0.  4.  4.  0.  0. -4.]
 [ 0.  0.  0.  4.  4.  0.  0. -4.]
 [ 0.  0.  0.  3.  3.  0.  0. -3.]]

strongest response at column index: 3  (the edge sits between columns 3 and 4)


### Example 2 — A full conv block + the weight-sharing payoff

A real conv layer applies *many* filters, then a nonlinearity, then pooling. Below we run **conv → ReLU → 2×2 max-pool** with three random filters and watch the shapes flow. Then we count parameters against an equivalent fully-connected layer to see *why* convolution scales: the same handful of weights is reused across every spatial position.

In [3]:
rng = np.random.default_rng(0)

def conv_layer(image, filters, padding=1):
    """Apply each (kH,kW) filter, stack the feature maps into channels."""
    return np.stack([conv2d(image, f, padding=padding) for f in filters])

def relu(x):
    return np.maximum(0, x)

def max_pool2d(fmap, size=2):
    """Max-pool each channel with a non-overlapping size x size window."""
    C, H, W = fmap.shape
    out = np.zeros((C, H // size, W // size))
    for c in range(C):
        for i in range(H // size):
            for j in range(W // size):
                out[c, i, j] = fmap[c, i*size:i*size+size, j*size:j*size+size].max()
    return out

filters = rng.normal(0, 0.5, (3, 3, 3))          # 3 filters, each 3x3
maps   = conv_layer(image, filters)              # (3, 8, 8)
acts   = relu(maps)                              # nonlinearity, same shape
pooled = max_pool2d(acts, size=2)                # (3, 4, 4)

print("input        ", image.shape)
print("after conv   ", maps.shape,   "(3 feature-map channels)")
print("after relu   ", acts.shape)
print("after pool   ", pooled.shape, "(half the spatial size)")

# Parameter count: conv layer vs. a dense layer producing the same number of outputs
K, C_in, C_out = 3, 1, 3
conv_params  = K*K*C_in*C_out + C_out                       # weights + biases
H = W = 8
dense_params = (H*W) * (C_out*H*W) + C_out*H*W              # fully-connected equivalent
print(f"\nconv layer params : {conv_params:>8,d}   (weights shared across all 64 positions)")
print(f"dense equivalent  : {dense_params:>8,d}   ({dense_params//conv_params:,}x more)")

input         (8, 8)
after conv    (3, 8, 8) (3 feature-map channels)
after relu    (3, 8, 8)
after pool    (3, 4, 4) (half the spatial size)

conv layer params :       30   (weights shared across all 64 positions)
dense equivalent  :   12,480   (416x more)


### Example 3 — The idiomatic PyTorch API (gated)

In practice you never hand-roll the loops above — `torch.nn.Conv2d` does it (vectorized, on GPU, with autograd). This cell shows the real API and the output-shape arithmetic. It runs only if you set `RUN_TORCH=1` and have Torch installed; otherwise it prints the reference snippet so the notebook still executes cleanly.

In [4]:
import os

TORCH_SNIPPET = """
import torch, torch.nn as nn

# A tiny LeNet-style classifier head for 1x28x28 grayscale images.
net = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1),  # 1x28x28 -> 8x28x28
    nn.ReLU(),
    nn.MaxPool2d(2),                                                      # 8x28x28 -> 8x14x14
    nn.Conv2d(8, 16, kernel_size=3, padding=1),                          # -> 16x14x14
    nn.ReLU(),
    nn.MaxPool2d(2),                                                      # -> 16x7x7
    nn.Flatten(),
    nn.Linear(16 * 7 * 7, 10),                                           # -> 10 class logits
)

x = torch.randn(4, 1, 28, 28)          # (batch, channels, H, W)
logits = net(x)                        # (4, 10)
print(logits.shape)

# out = floor((W + 2P - K)/S) + 1   ->  (28 + 2*1 - 3)/1 + 1 = 28  (padding='same')
conv = net[0]
print("conv0 weight:", tuple(conv.weight.shape), " params:", conv.weight.numel() + conv.bias.numel())
"""

if os.getenv("RUN_TORCH") == "1":
    try:
        exec(TORCH_SNIPPET)
    except ImportError:
        print("PyTorch not installed — run `pip install torch` first.")
else:
    print("[skipped] set RUN_TORCH=1 (and `pip install torch`) to execute the PyTorch example.")
    print("Reference snippet:")
    print(TORCH_SNIPPET)

[skipped] set RUN_TORCH=1 (and `pip install torch`) to execute the PyTorch example.
Reference snippet:

import torch, torch.nn as nn

# A tiny LeNet-style classifier head for 1x28x28 grayscale images.
net = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1),  # 1x28x28 -> 8x28x28
    nn.ReLU(),
    nn.MaxPool2d(2),                                                      # 8x28x28 -> 8x14x14
    nn.Conv2d(8, 16, kernel_size=3, padding=1),                          # -> 16x14x14
    nn.ReLU(),
    nn.MaxPool2d(2),                                                      # -> 16x7x7
    nn.Flatten(),
    nn.Linear(16 * 7 * 7, 10),                                           # -> 10 class logits
)

x = torch.randn(4, 1, 28, 28)          # (batch, channels, H, W)
logits = net(x)                        # (4, 10)
print(logits.shape)

# out = floor((W + 2P - K)/S) + 1   ->  (28 + 2*1 - 3)/1 + 1 = 28  (padding='same')
conv = net[0]
print("conv0 weight:", tuple(conv.weig

## 6. Gotchas & Pitfalls

- **Shape arithmetic bites.** Forgetting padding shrinks maps by `K−1` each layer; a `nn.Linear` after the conv stack must match the flattened size exactly, or you get a runtime shape error. Compute `floor((W+2P−K)/S)+1` by hand, or print shapes as above.
- **Channel order conventions differ.** PyTorch is **NCHW** `(batch, channels, H, W)`; TensorFlow/Keras default to **NHWC**. Mixing them silently transposes your data.
- **"Convolution" is really cross-correlation.** DL libraries don't flip the kernel (true math convolution does). It doesn't matter for learning — the network just learns the flipped weights — but don't be surprised reading the source.
- **Forgetting to normalize inputs.** Feed raw 0–255 pixels and activations explode; scale to ~[0,1] or standardize per-channel. Pair conv layers with **BatchNorm** for stable, faster training.
- **Pooling away too much, too early.** Aggressive early downsampling destroys fine detail you can never recover — bad for segmentation/detection. Modern nets favor strided convs and keep resolution longer (see [`u-net`](u-net.ipynb) skip connections).
- **Tiny datasets, huge nets.** CNNs overfit fast. Use data augmentation (flips, crops, color jitter) and **transfer learning** from a pretrained backbone instead of training from scratch.
- **Very deep plain stacks don't train.** Beyond ~20 layers, gradients degrade; **residual connections** ([`resnet`](resnet.ipynb)) are what made 50–150 layer CNNs trainable.
- **Receptive field too small for the task.** If objects span more of the image than any unit can see, the model literally cannot integrate the evidence — add depth, larger/dilated kernels, or pooling.

## 7. When to Use vs Alternatives

| Option | Best at | Trade-off vs CNN |
|---|---|---|
| **CNN** (this notebook) | Images & grid data; strong with limited data thanks to built-in locality + weight-sharing priors | Local by construction — long-range/global relations need depth or extra mechanisms |
| **Vision Transformer** ([`vision-transformer`](vision-transformer.ipynb)) | Large datasets, global context, multimodal pairing | Few built-in priors ⇒ data-hungry; quadratic attention cost; often needs heavy pretraining |
| **ResNet / modern CNN** ([`resnet`](resnet.ipynb)) | Very deep CNNs that actually train; the standard strong image baseline | A *kind of* CNN — residual blocks, not a different family |
| **U-Net** ([`u-net`](u-net.ipynb)) | Dense pixel-wise output: segmentation, image-to-image | Encoder-decoder with skips; overkill for plain classification |
| **MLP / fully-connected** | Tabular, no spatial structure | On images: explodes in parameters, ignores locality, overfits |
| **RNN / 1D-CNN** ([`rnn`](rnn.ipynb), [`lstm`](lstm.ipynb)) | Sequences/time series | A 1D CNN *is* a strong sequence model; RNNs handle variable-length/causal streaming |

**Rule of thumb:** images or local-grid data, especially with a modest dataset → start with a CNN (a pretrained ResNet is a great default). Massive data, need for global context, or multimodal goals → consider a Vision Transformer. Hybrids (conv stem + attention) are common and often best of both.

## 8. Resources

- **CS231n: Convolutional Neural Networks for Visual Recognition** — the canonical course notes, still the clearest explanation of conv arithmetic and architectures: https://cs231n.github.io/convolutional-networks/
- **PyTorch `nn.Conv2d` docs** — exact API, padding modes, and the output-shape formula: https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
- **"A guide to convolution arithmetic for deep learning"** (Dumoulin & Visin) — every stride/padding/dilation case with animations: https://arxiv.org/abs/1603.07285
- **Deep Residual Learning (ResNet)** — He et al., the paper that unlocked very deep CNNs: https://arxiv.org/abs/1512.03385
- **ImageNet Classification with Deep CNNs (AlexNet)** — Krizhevsky et al., the result that started the modern CNN era: https://papers.nips.cc/paper/2012/hash/c399862d3b9d6b76c8436e924a68c45b-Abstract.html